# Document vs Photo ONNX Classification

This notebook runs `mobilenetv2_doc_photo_quant.onnx` on `1.jpeg`.

Class mapping:
- `0`: document
- `1`: photo

In [37]:
from pathlib import Path

import numpy as np
import onnxruntime as ort
from PIL import Image

# MODEL_PATH = Path("mobilenetv2_doc_photo_quant.onnx")
MODEL_PATH = Path("/Users/mac/Downloads/mobilenet_v3_small_quant.onnx")
IMAGE_PATH = Path("/Users/mac/Desktop/Screenshot 2026-05-30 at 11.20.07 AM.png")

LABELS = {
    0: "document",
    1: "photo",
}

print("Model exists:", MODEL_PATH.exists(), MODEL_PATH.resolve())
print("Image exists:", IMAGE_PATH.exists(), IMAGE_PATH)

Model exists: True /Users/mac/Downloads/mobilenet_v3_small_quant.onnx
Image exists: True /Users/mac/Desktop/Screenshot 2026-05-30 at 11.20.07 AM.png


In [38]:
session = ort.InferenceSession(str(MODEL_PATH), providers=["CPUExecutionProvider"])

input_info = session.get_inputs()[0]
output_info = session.get_outputs()[0]

print("Input name:", input_info.name)
print("Input shape:", input_info.shape)
print("Input type:", input_info.type)
print("Output name:", output_info.name)
print("Output shape:", output_info.shape)
print("Output type:", output_info.type)

Input name: input
Input shape: [1, 3, 224, 224]
Input type: tensor(float)
Output name: output
Output shape: [1, 2]
Output type: tensor(float)


In [39]:
def preprocess_image(image_path: Path) -> np.ndarray:
    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224), Image.BILINEAR)

    array = np.asarray(image, dtype=np.float32) / 255.0

    # Standard MobileNetV2/ImageNet normalization.
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    array = (array - mean) / std

    # Convert HWC RGB image to NCHW batch tensor: [1, 3, 224, 224].
    array = np.transpose(array, (2, 0, 1))[None, ...]
    return array.astype(np.float32)


input_tensor = preprocess_image(IMAGE_PATH)
print(input_tensor.shape, input_tensor.dtype)

(1, 3, 224, 224) float32


In [40]:
def softmax(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits)
    exp = np.exp(logits)
    return exp / np.sum(exp)


input_name = session.get_inputs()[0].name
raw_output = session.run(None, {input_name: input_tensor})[0][0]
probabilities = softmax(raw_output)
predicted_class = int(np.argmax(probabilities))

print("Raw output:", raw_output)
print("Probabilities:")
for class_id, probability in enumerate(probabilities):
    print(f"  {class_id} - {LABELS[class_id]}: {probability:.4f}")

print(f"\nPrediction: {predicted_class} - {LABELS[predicted_class]}")

Raw output: [-0.08170175 -0.08471471]
Probabilities:
  0 - document: 0.5008
  1 - photo: 0.4992

Prediction: 0 - document
